# Distance Measures Comparison

**Purpose:** Compare ultrametric and tree distance measures across phases and patients.
**Inputs:** SEEG `.mat` files in `data/stereoeeg_patients/` and channel metadata CSVs.
**Outputs:** Figures in `data/figures/distance_measures_comparison/`.
**Date:** 2025-12-11


In [ ]:
# %% Configuration
CONFIG = {
    "patients": ["Pat_02", "Pat_03"],
    "phases": ["rsPre", "taskLearn", "taskTest", "rsPost"],
    "bands": ["delta", "theta", "alpha", "beta", "low_gamma", "high_gamma"],
    "output_dir": "figures/distance_measures_comparison",
    "correlation_protocol": {"filter_type": "abs", "spectral_cleaning": False, "threshold": 0},
    "filter_order": 1,
    "linkage_method": "average",
    "distance_metric": "euclidean",
}


In [ ]:
# %% Setup
from lrgsglib import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")

import matplotlib.pyplot as plt
import numpy as np

from lrg_eegfc.notebook import *
from lrg_eegfc.utils.corrmat.structures import compute_structures_for_patient
from lrgsglib.utils.basic.linalg import (
    ultrametric_matrix_distance,
    ultrametric_scaled_distance,
    ultrametric_rank_correlation,
    ultrametric_quantile_rmse,
    ultrametric_distance_permutation_robust,
    tree_robinson_foulds_distance,
    tree_cophenetic_correlation,
    tree_baker_gamma,
    tree_fowlkes_mallows_index,
)

path_figs = setup_notebook(CONFIG["output_dir"])


## 1. Load Data

Load data for the selected patients and phases.

In [ ]:
data_dict, int_label_map = load_data_dict(
    pat_list=CONFIG["patients"],
    phase_labels=CONFIG["phases"],
)
pin_labels_by_pat = {pat: int_label_map[pat]["label"] for pat in CONFIG["patients"]}
print("✓ Data loaded")


## 2. Compute Ultrametric Structures

Build ultrametric and linkage matrices for every patient, phase, and band.

In [ ]:
ultra_by_pat = {}
linkage_by_pat = {}
condensed_by_pat = {}

for pat in CONFIG["patients"]:
    print(f"Computing ultrametric structures for {pat} ...")
    U, Z, D = compute_structures_for_patient(
        data_dict,
        pat,
        int_label_map,
        bands=CONFIG["bands"],
        phases=CONFIG["phases"],
        correlation_protocol=CONFIG["correlation_protocol"],
        filter_order=CONFIG["filter_order"],
        linkage_method=CONFIG["linkage_method"],
    )
    ultra_by_pat[pat] = U
    linkage_by_pat[pat] = Z
    condensed_by_pat[pat] = D

print("✓ Ultrametric structures computed")


## 3. Compute Distance Measures

Compute nine distance measures for each band and patient.

In [ ]:
def _safe_same_shape(U1, U2):
    if U1 is None or U2 is None:
        return False
    if not hasattr(U1, "shape") or not hasattr(U2, "shape"):
        return False
    return U1.shape == U2.shape

def _measure_ultrametric_matrix_distance(U1, U2, *_):
    if not _safe_same_shape(U1, U2):
        return np.nan
    return ultrametric_matrix_distance(U1, U2, metric=CONFIG["distance_metric"])

def _measure_ultrametric_scaled_distance(U1, U2, *_):
    if not _safe_same_shape(U1, U2):
        return np.nan
    return ultrametric_scaled_distance(
        U1, U2, metric=CONFIG["distance_metric"], scale="log", normalize=True
    )

def _measure_ultrametric_rank_distance(U1, U2, *_):
    if not _safe_same_shape(U1, U2):
        return np.nan
    corr = ultrametric_rank_correlation(U1, U2, method="spearman")
    return 1.0 - corr if corr is not None else np.nan

def _measure_ultrametric_quantile_rmse(U1, U2, *_):
    if not _safe_same_shape(U1, U2):
        return np.nan
    return ultrametric_quantile_rmse(U1, U2, scale="log")

def _measure_permutation_robust(U1, U2, Z1, Z2, D1, D2, labels):
    if Z1 is None or Z2 is None or D1 is None or D2 is None:
        return np.nan
    return ultrametric_distance_permutation_robust(
        Z1, Z2, D1, D2, labels, metric=CONFIG["distance_metric"]
    )

def _measure_robinson_foulds(U1, U2, Z1, Z2, *_):
    if Z1 is None or Z2 is None:
        return np.nan
    return tree_robinson_foulds_distance(Z1, Z2, normalized=True)

def _measure_cophenetic_distance(U1, U2, Z1, Z2, *_):
    if Z1 is None or Z2 is None:
        return np.nan
    corr = tree_cophenetic_correlation(Z1, Z2)
    return 1.0 - corr if corr is not None else np.nan

def _measure_baker_gamma_distance(U1, U2, Z1, Z2, *_):
    if Z1 is None or Z2 is None:
        return np.nan
    gamma = tree_baker_gamma(Z1, Z2)
    return 1.0 - gamma if gamma is not None else np.nan

def _measure_fowlkes_mallows_distance(U1, U2, Z1, Z2, *_):
    if Z1 is None or Z2 is None:
        return np.nan
    fm = tree_fowlkes_mallows_index(Z1, Z2)
    return 1.0 - fm if fm is not None else np.nan

measure_defs = [
    {
        "key": "ultrametric_matrix_distance",
        "label": "Ultrametric Matrix Distance",
        "cbar": "Ultrametric matrix distance",
        "fn": _measure_ultrametric_matrix_distance,
    },
    {
        "key": "ultrametric_scaled_distance_log",
        "label": "Ultrametric Scaled Distance (log)",
        "cbar": "Ultrametric scaled distance (log)",
        "fn": _measure_ultrametric_scaled_distance,
    },
    {
        "key": "ultrametric_rank_correlation_spearman",
        "label": "Ultrametric Rank Distance (1 - Spearman)",
        "cbar": "Distance (1 - Spearman correlation)",
        "fn": _measure_ultrametric_rank_distance,
    },
    {
        "key": "ultrametric_quantile_rmse_log",
        "label": "Ultrametric Quantile RMSE (log)",
        "cbar": "Ultrametric quantile RMSE (log)",
        "fn": _measure_ultrametric_quantile_rmse,
    },
    {
        "key": "ultrametric_distance_permutation_robust",
        "label": "Ultrametric Distance (Permutation Robust)",
        "cbar": "Permutation-robust distance",
        "fn": _measure_permutation_robust,
    },
    {
        "key": "tree_robinson_foulds",
        "label": "Robinson-Foulds Distance",
        "cbar": "Robinson-Foulds distance (normalized)",
        "fn": _measure_robinson_foulds,
    },
    {
        "key": "tree_cophenetic_distance",
        "label": "Cophenetic Correlation (distance)",
        "cbar": "Distance (1 - cophenetic correlation)",
        "fn": _measure_cophenetic_distance,
    },
    {
        "key": "tree_baker_gamma_distance",
        "label": "Baker's Gamma (distance)",
        "cbar": "Distance (1 - Baker's gamma)",
        "fn": _measure_baker_gamma_distance,
    },
    {
        "key": "tree_fowlkes_mallows_distance",
        "label": "Fowlkes-Mallows Distance",
        "cbar": "Distance (1 - Fowlkes-Mallows)",
        "fn": _measure_fowlkes_mallows_distance,
    },
]

measure_results = {}
plot_specs = {}

for measure in measure_defs:
    key = measure["key"]
    plot_specs[key] = {
        "label": measure["label"],
        "cbar": measure["cbar"],
    }
    measure_results[key] = {band: {} for band in CONFIG["bands"]}

    for band in CONFIG["bands"]:
        for pat in CONFIG["patients"]:
            U_band = ultra_by_pat[pat][band]
            Z_band = linkage_by_pat[pat][band]
            D_band = condensed_by_pat[pat][band]
            labels = pin_labels_by_pat[pat]

            def compute_fn(pi, pj, U_band=U_band, Z_band=Z_band, D_band=D_band, labels=labels):
                U1 = U_band.get(pi)
                U2 = U_band.get(pj)
                Z1 = Z_band.get(pi)
                Z2 = Z_band.get(pj)
                D1 = D_band.get(pi)
                D2 = D_band.get(pj)
                return measure["fn"](U1, U2, Z1, Z2, D1, D2, labels)

            measure_results[key][band][pat] = compute_phase_distance_matrix(
                CONFIG["phases"],
                compute_fn,
                diag_value=0.0,
                patient=pat,
                band=band,
            )
        print(f"Band {band}: ✓ {measure['label']} computed")

print("✓ All measures computed")


## 4. Plot Measures Across Patients

Save band-by-band heatmaps for each measure.

In [ ]:
def plot_measure_across_patients(measure_key, measure_label, cbar_label):
    outdir = path_figs / measure_key
    outdir.mkdir(parents=True, exist_ok=True)

    for band in CONFIG["bands"]:
        vmax = 0.0
        for pat in CONFIG["patients"]:
            M = measure_results[measure_key][band].get(pat)
            if M is not None and np.isfinite(M).any():
                vmax = max(vmax, np.nanmax(M))

        if vmax == 0.0:
            print(f"[SKIP] No finite values for {measure_label} / {band}")
            continue

        fig, axes = plt.subplots(
            1,
            len(CONFIG["patients"]),
            figsize=(4 * len(CONFIG["patients"]) + 2, 4),
            constrained_layout=True,
        )
        if len(CONFIG["patients"]) == 1:
            axes = [axes]

        cmap = plt.cm.viridis.copy()
        cmap.set_bad(color="lightgray")

        for ax, pat in zip(axes, CONFIG["patients"]):
            M = measure_results[measure_key][band][pat]
            im = ax.imshow(M, vmin=0.0, vmax=vmax, cmap=cmap, aspect="equal")
            ax.set_title(pat)
            ax.set_xticks(range(len(CONFIG["phases"])))
            ax.set_yticks(range(len(CONFIG["phases"])))
            ax.set_xticklabels(CONFIG["phases"], rotation=45, ha="right")
            ax.set_yticklabels(CONFIG["phases"])
            for spine in ax.spines.values():
                spine.set_visible(False)

        cbar = fig.colorbar(im, ax=axes, shrink=0.85)
        cbar.set_label(cbar_label)
        fig.suptitle(f"{measure_label} — {band} band", fontsize=14)

        outfile = outdir / f"{band}.png"
        fig.savefig(outfile, dpi=200, bbox_inches="tight")
        plt.show()
        print(f"Saved: {outfile}")

for key, spec in plot_specs.items():
    print(f"
=== {spec['label']} ===")
    plot_measure_across_patients(key, spec["label"], spec["cbar"])

print("✓ All figures generated")


## 5. Cross-Patient Consistency and Ranking

Rank measures based on cross-patient consistency metrics.

In [ ]:
if len(CONFIG["patients"]) < 2:
    print("Need at least two patients for cross-patient comparison.")
else:
    rankings = rank_distance_measures(
        measure_results,
        patients=CONFIG["patients"],
        bands=CONFIG["bands"],
    )

    print("
RANKING: MOST CONSISTENT MEASURES")
    print("-" * 60)
    ranked = sorted(
        rankings["scores"].items(),
        key=lambda x: x[1] if np.isfinite(x[1]) else -np.inf,
        reverse=True,
    )
    for idx, (key, score) in enumerate(ranked, 1):
        label = plot_specs[key]["label"]
        print(f"{idx:2d}. {label:45s} {score:8.4f}")

    best_measure = rankings["best_measure"]
    if best_measure is not None:
        print("
BEST MEASURE:")
        print(f"  → {plot_specs[best_measure]['label']}")

    # Simple score plot
    labels = [plot_specs[k]["label"] for k, _ in ranked]
    scores = [s for _, s in ranked]

    fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(labels))))
    ax.barh(labels, scores, color="steelblue")
    ax.set_xlabel("Combined consistency score")
    ax.set_title("Cross-Patient Consistency (Higher is Better)")
    ax.axvline(0, color="black", linewidth=0.5)
    ax.grid(axis="x", alpha=0.3)

    outfile = path_figs / "consistency_ranking.png"
    fig.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {outfile}")
